# Vector stores and semantic search



In [109]:
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [110]:
import torch

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata

    def __repr__(self):
        return f"Document(text={self.text}, metadata={self.metadata})"
        

class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document

    def __repr__(self):
        return f"Text: {self.document.text}\nScore: {self.score}"


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.documents = []
        self.embeddings = None
        self.model = embedding_model

    def add_documents(self, documents: list[Document]):
        new_embeddings = self.model.encode(
            [doc.text for doc in documents],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

        self.documents.extend(documents)

    def get_n_documents(self, n: int) -> list[Document]:
        return self.documents[:n]

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        embeded_query = self.model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
        scores = embeded_query @ self.embeddings.T

        sorted_scores, sorted_idx = torch.sort(scores, descending=True)
        top_scores = sorted_scores[:top_k]
        top_indices = sorted_idx[:top_k]

        return [SearchResult(score.item(), self.documents[idx]) for score, idx in zip(top_scores, top_indices)]

### Load dataset

In [111]:
import csv

In [112]:
documents = []

with open('data/animal-fun-facts-dataset.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for i, row in enumerate(reader):
        text = str(row.pop('text')) 
        documents.append(Document(text=text, metadata=row))

print(f"Loaded {len(documents)} documents.\n")

for i, doc in enumerate(documents):
    if (i >= 5):
        break
    print(f"Document {i+1}:")
    print(doc.text)
    print(doc.metadata)
    print()


Loaded 7734 documents.

Document 1:
Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 2:
Aardvarks
have rather primitive brains that are very small for the size of the
animal. Some have suggested they are not particularly bright....
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 3:
Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel.
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 4:
The aardvarks Latin family name "Tubulidentata" means "tube toothed"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-f

### Initialize Vector Store

In [113]:
model = SentenceTransformer("all-MiniLM-L6-v2")
store = VectorStore(model)
store.add_documents(documents)
print(model.device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3855.75it/s]


cuda:0


In [114]:
print(store.get_n_documents(3))

[Document(text=Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters", metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}), Document(text=Aardvarks
have rather primitive brains that are very small for the size of the
animal. Some have suggested they are not particularly bright...., metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}), Document(text=Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel., metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'})]


### Making queries

In [115]:
store.search("What is the fastest animal?")

[Text: Fastest animal on Earth
 Score: 0.9126603603363037,
 Text: The fastest creatures on the planet!
 Score: 0.8299250602722168,
 Text: The fastest land mammal in the world!
 Score: 0.8021312952041626,
 Text: The fastest species of primate in the world!
 Score: 0.7151623368263245,
 Text: Patagonian mara are the worlds fastest rodent
 .
 On that topic, they can really move! Plains animals often use speed to their advantage (think cheetahs and gazelles), but the mara is a record-holder in that department.
 Score: 0.7100554704666138]

In [116]:
store.search("Are there any animals that can fly?")

[Text: They don’t fly, they glide.
 The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.
 Score: 0.6900345087051392,
 Text: Not all birds are able to fly!
 Score: 0.6892896890640259,
 Text: Bats are the only mammals with wings, and the only ones that can truly fly
 Score: 0.6677284836769104,
 Text: They rarely fly..
 They move around on foot most of the time, only taking to the air to reach their nests or for courtship displays.
 Score: 0.6263091564178467,
 Text: Bats are the world's only flying mammals. Other mammals may glide through the air, but bats flap their wings and fly.
 Score: 0.6174590587615967]

In [118]:
store.search("Are golden retrievers good pets?")

[Text: Golden Pyrenees make great therapy dogs due to their intelligence and gentle nature.
 Score: 0.6628046035766602,
 Text: They are not great pets.
 In the USA and some other parts of the world they are kept as pets, however they are extremely strong and can be very dangerous as they get older. They are wild animals, and not domesticated.
 Score: 0.5993548631668091,
 Text: These dogs are very intelligent and are great with children.
 Score: 0.5695565938949585,
 Text: Although their coats can get incredibly light in color, golden retrievers never have purely white coats.
 Score: 0.565988302230835,
 Text: The Golden Shepherds were first recognized by the International Designer Canine Registry in 2009.
 Score: 0.5427731275558472]

In [119]:
store.search("Are crabs immortal?")

[Text: It may have the longest live span among crabs.
 Commonly known crabs—such as Dungeness crabs, king crabs, and snow crabs—live for several decades (between 10 to 30 years).
 Score: 0.7234090566635132,
 Text: There is a limit as to how large their bodies can grow.
 The carapace, or hard upper shell, of Japanese spider crabs cap at a particular size once it reaches adulthood. Their legs, however, keep on growing and elongating.
 Score: 0.5835776925086975,
 Text: Closely related to crabs and lobsters!
 Score: 0.5463244915008545,
 Text: Closely related to crabs and lobsters!
 Score: 0.5463244915008545,
 Text: Hermit crabs will eventually outgrow their shells. At such times, they'd line up according to size to swap shells. Shells too big or too small are rejected. This chain reaction is called "vacancy chain." Vacancy chain is a way for these crustaceans to survive while sharing limited resources.
 Score: 0.5435934662818909]

In [121]:
store.search("Hay un animal que hable español?")

[Text: They’re named after their hands.
 Both the Quebeqi and the Spanish-speaking colonists who named raccoons independently referred to the hands of the animal. The word aroughcoune roughly translates to “the one that scratches with his hands”, and mapachtli in Spanish refers to “the animal with the hands”.
 Score: 0.4280066192150116,
 Text: The resplendent quetzal is a sacred symbol in Mesoamerica and Guatemala's national bird, pictured on the country's flag. They favor eating fruit in the avocado family, eating them whole before regurgitating the pits. Essentially making them the avocado "gardeners" of their forest habitats.
 Score: 0.41232359409332275,
 Text: Axolotl is a Mexican word that translates to "Water dog"
 Score: 0.4037732481956482,
 Text: A hairless breed of dog!
 Score: 0.39489513635635376,
 Text: The second largest animal on the land!
 Score: 0.39259788393974304]

## Part II: Filtering by metadata

In [117]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass